# Hyperparameter Tuning — Complete Guide

- **GridSearchCV**: Exhaustive search over parameter grid
- **RandomizedSearchCV**: Random sampling (faster for large grids)
- **Optuna**: Bayesian optimization (intelligent search)
- **Halving Search**: Successive halving for efficiency
- **Visualization**: Parameter impact analysis

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split, GridSearchCV, RandomizedSearchCV,
                                     cross_val_score, StratifiedKFold)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from scipy.stats import randint, uniform

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    print(f'Optuna: {optuna.__version__}')
except ImportError:
    print('Install: pip install optuna')

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Setup
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 1. GridSearchCV — Exhaustive Search

**Pros**: Tests every combination; guaranteed to find the best in the grid  
**Cons**: Exponentially expensive with more parameters  
**Use when**: Small parameter space, few hyperparameters

In [ ]:
# GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

total_combinations = 1
for v in param_grid.values():
    total_combinations *= len(v)
print(f'Total combinations: {total_combinations}')
print(f'With 5-fold CV: {total_combinations * 5} model fits')

start = time.time()
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5, scoring='accuracy', n_jobs=-1, return_train_score=True
)
grid_search.fit(X_train, y_train)
grid_time = time.time() - start

print(f'\nBest Parameters: {grid_search.best_params_}')
print(f'Best CV Score:   {grid_search.best_score_:.4f}')
print(f'Test Score:      {grid_search.score(X_test, y_test):.4f}')
print(f'Time:            {grid_time:.2f}s')

## 2. RandomizedSearchCV — Random Sampling

**Pros**: Much faster; can use continuous distributions  
**Cons**: May miss the absolute best; results vary by n_iter  
**Use when**: Large parameter space, many hyperparameters

In [ ]:
# RandomizedSearchCV with distributions
param_distributions = {
    'n_estimators': randint(50, 500),
    'max_depth': [3, 5, 7, 10, 15, None],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}

start = time.time()
random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions, n_iter=100, cv=5, scoring='accuracy',
    n_jobs=-1, random_state=42, return_train_score=True
)
random_search.fit(X_train, y_train)
random_time = time.time() - start

print(f'Best Parameters: {random_search.best_params_}')
print(f'Best CV Score:   {random_search.best_score_:.4f}')
print(f'Test Score:      {random_search.score(X_test, y_test):.4f}')
print(f'Time:            {random_time:.2f}s')

## 3. Optuna — Bayesian Optimization

**Pros**: Intelligent search (learns from past trials); pruning; visualization  
**Cons**: Slightly more setup  
**Use when**: You want the best results efficiently

In [ ]:
def objective(trial):
    """Optuna objective function for RandomForest."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'random_state': 42
    }
    
    model = RandomForestClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    return scores.mean()

start = time.time()
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=True)
optuna_time = time.time() - start

print(f'\nBest Parameters: {study.best_params}')
print(f'Best CV Score:   {study.best_value:.4f}')

# Test with best params
best_rf = RandomForestClassifier(**study.best_params, random_state=42)
best_rf.fit(X_train, y_train)
print(f'Test Score:      {best_rf.score(X_test, y_test):.4f}')
print(f'Time:            {optuna_time:.2f}s')

In [ ]:
# Optuna visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Optimization history
trials = study.trials
values = [t.value for t in trials]
best_values = [max(values[:i+1]) for i in range(len(values))]
axes[0].plot(values, alpha=0.3, label='Trial Score')
axes[0].plot(best_values, color='red', linewidth=2, label='Best So Far')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('CV Accuracy')
axes[0].set_title('Optuna Optimization History')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Parameter importance
param_importances = optuna.importance.get_param_importances(study)
params_sorted = sorted(param_importances.items(), key=lambda x: x[1])
axes[1].barh([p[0] for p in params_sorted], [p[1] for p in params_sorted], color='steelblue')
axes[1].set_xlabel('Importance')
axes[1].set_title('Hyperparameter Importance (Optuna)')

plt.tight_layout()
plt.show()

## 4. Method Comparison

In [ ]:
comparison = pd.DataFrame({
    'Method': ['GridSearchCV', 'RandomizedSearchCV', 'Optuna (Bayesian)'],
    'Best CV Score': [grid_search.best_score_, random_search.best_score_, study.best_value],
    'Test Score': [grid_search.score(X_test, y_test), random_search.score(X_test, y_test), best_rf.score(X_test, y_test)],
    'Time (s)': [grid_time, random_time, optuna_time],
    'Evaluations': [total_combinations * 5, 100 * 5, 100 * 5]
})
print(comparison.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71']
for ax, col in zip(axes, ['Test Score', 'Time (s)']):
    bars = ax.bar(comparison['Method'], comparison[col], color=colors, edgecolor='black')
    for bar, val in zip(bars, comparison[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.4f}' if col == 'Test Score' else f'{val:.1f}s',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_title(col)
    ax.grid(axis='y', alpha=0.3)
plt.suptitle('Hyperparameter Tuning Methods Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## 5. When to Use Which?

| Method | Speed | Quality | Use Case |
|--------|-------|---------|----------|
| **GridSearchCV** | Slowest | Exhaustive | Small grids (< 100 combinations) |
| **RandomizedSearchCV** | Fast | Good | Large grids, quick baseline |
| **Optuna** | Smart | **Best** | Production tuning, complex spaces |
| **HalvingGridSearch** | Fastest grid | Good | Large datasets, budget-limited |

### Practical Workflow
1. Start with **RandomizedSearchCV** (n_iter=50) to narrow the range
2. Use **GridSearchCV** on the narrowed range for fine-tuning
3. Or use **Optuna** for the best of both worlds